# 📘 COMPAS Dataset — Complete Study Notes

**A beginner-friendly walkthrough of the ProPublica COMPAS recidivism dataset**

Welcome! This notebook is your complete study guide for the **COMPAS dataset** — one of the two datasets used in our FairML project (the other is the German Credit dataset, covered in its own notebook).

By the end of this notebook you will understand:

1. **What the COMPAS dataset is** and why it became famous in fairness research
2. **What every column means**, in plain language
3. **The shape and structure** of the data (rows, columns, types, missing values, duplicates)
4. **Exploratory Data Analysis (EDA)** — what the data actually looks like
5. **Fairness concerns** — which attributes are "protected" and why this dataset is a landmark case of algorithmic bias
6. **The exact preprocessing workflow** our project (`Compas.py`) uses before training models

💡 **How to use this notebook:** Run each code cell in order (press `Shift + Enter`). Before every code cell there is an explanation of *what we are about to do*, and after it an explanation of *what the output means*.


---
## 1. Dataset Overview

### What is COMPAS?

**COMPAS** stands for **C**orrectional **O**ffender **M**anagement **P**rofiling for **A**lternative **S**anctions. It is a commercial software tool used by U.S. courts to predict how likely a criminal defendant is to **re-offend (recidivate)** — that is, commit another crime in the future.

In 2016, the investigative journalism organization **ProPublica** collected data on ~7,000 defendants from Broward County, Florida, who were scored by COMPAS in 2013–2014. They then followed what actually happened to those people for **two years** and published the data. Their famous finding: the algorithm's mistakes were not evenly distributed across races — Black defendants were almost twice as likely to be *incorrectly* flagged as high-risk compared to white defendants.

### What problem does this dataset help solve?

The prediction task is: **given information about a defendant (age, prior crimes, charge details, demographics), predict whether they will re-offend within two years.**

This is a **binary classification** problem (two possible answers: yes or no). In our FairML project, we don't just try to predict accurately — we try to predict **fairly**, by adding fairness penalties to the model's training loss.

### What does each row represent?

**One row = one defendant** who was scored by COMPAS in Broward County. Each row contains their demographics, criminal history, the charge they were arrested for, the COMPAS scores they received, and — crucially — whether they actually re-offended within two years.

### What is the target variable?

The target (the thing we predict) is **`two_year_recid`**:

| Value | Meaning |
|-------|---------|
| `1` | The person **did** re-offend within two years (the *adverse* outcome) |
| `0` | The person did **not** re-offend within two years |


---
## 2. Setup — Importing Our Tools

**What we're about to do:** Import the Python libraries we need. Think of these as our toolbox:

- **pandas** — for loading and manipulating tables of data (like Excel, but in code)
- **numpy** — for fast math on arrays of numbers
- **matplotlib & seaborn** — for drawing charts
- **warnings** — just to silence noisy messages that would clutter our outputs

We also define a small **fixed color palette** so every chart in this notebook uses the same, colorblind-safe colors. Using consistent colors makes charts easier to read and compare.


In [ ]:
# Import the libraries (our data-science toolbox)
import pandas as pd              # tables of data
import numpy as np               # numerical arrays and math
import matplotlib.pyplot as plt  # basic plotting
import seaborn as sns            # prettier statistical plots (built on matplotlib)

import warnings
warnings.filterwarnings("ignore")  # hide non-critical warning messages

# --- A fixed, colorblind-safe palette used for EVERY chart in this notebook ---
# (These are from seaborn's "colorblind" palette, designed to stay
#  distinguishable for people with color-vision deficiencies.)
COLORS = sns.color_palette("colorblind")
C_NO   = COLORS[0]   # blue   -> "did NOT re-offend" (label 0)
C_YES  = COLORS[1]   # orange -> "DID re-offend"     (label 1)
C_MAIN = COLORS[0]   # blue   -> single-series charts

sns.set_theme(style="whitegrid", palette="colorblind")  # clean chart style
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded successfully!")


**What the output means:** If you see `Libraries loaded successfully!` everything imported fine. In Google Colab all of these libraries come pre-installed, so no `pip install` is needed.


---
## 3. Loading the Data

**What we're about to do:** Download the COMPAS dataset directly from ProPublica's public GitHub repository. This is the **exact same URL our project script `Compas.py` uses**, so we are studying exactly the data our models are trained on.

The file is a **CSV** (Comma-Separated Values) — a plain-text table. `pd.read_csv()` downloads it and turns it into a **DataFrame** (pandas' name for a table).


In [ ]:
# The official ProPublica data (same URL used in Compas.py)
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"

# Download the CSV and load it into a DataFrame called `data`
data = pd.read_csv(url)

# Show the first 5 rows to get a first impression
data.head()


**What the output means:** You should see a table with the first 5 defendants. Each **row is one person**; each **column is one piece of information** about them. Notice the mix of:
- **numbers** (like `age`, `priors_count`),
- **text/categories** (like `race`, `sex`, `c_charge_degree`),
- **dates** (like `compas_screening_date`),
- and many columns that are **empty (NaN)** for some people — e.g. the `r_*` columns are only filled in if the person actually re-offended.

Don't worry about understanding all 50+ columns yet — the next section explains them, and our project only uses 9 of them.


---
## 4. Column-by-Column Explanation

The raw file has **53 columns**. They fall into natural groups. First, the ⭐ **9 features + 1 target our project actually uses**, then the rest.

### ⭐ Columns used in our FairML project

| Column | Type | What it means | Why it matters |
|--------|------|---------------|----------------|
| `age` | Numerical (integer) | The defendant's age in years at screening | Younger people statistically re-offend more often — one of the strongest predictors |
| `juv_fel_count` | Numerical (integer) | Number of **juvenile felony** charges (serious crimes committed as a minor) | Early serious offending signals higher risk |
| `juv_misd_count` | Numerical (integer) | Number of **juvenile misdemeanor** charges (less serious crimes as a minor) | Same idea, but for minor offenses |
| `juv_other_count` | Numerical (integer) | Other juvenile charges that are neither felonies nor misdemeanors | Completes the juvenile-history picture |
| `priors_count` | Numerical (integer) | Total number of **prior adult crimes** | Usually the single most predictive feature: past behavior predicts future behavior |
| `c_charge_degree` | Categorical (text) | Degree of the **c**urrent charge: `F` = Felony (serious), `M` = Misdemeanor (minor) | Seriousness of the current offense |
| `race` | Categorical (text) | Race: African-American, Caucasian, Hispanic, Asian, Native American, Other | ⚠️ **Protected attribute** — our fairness analyses group people by race |
| `sex` | Categorical (text) | `Male` or `Female` | Also a protected attribute; men re-offend at higher rates in this data |
| `duration` *(engineered)* | Numerical | `end − start`: number of days the person was followed/observed in the study | Created by our own code (`Compas.py`). ⚠️ Strongly related to the target in a partly *mechanical* way — re-offending can cut the observation window short — so treat it with suspicion (details in the correlation section) |
| **`two_year_recid`** | **Target** (0/1) | Did the person re-offend within 2 years? `1` = yes, `0` = no | **This is what we predict** |

### Other column groups (present in the file, but NOT used by our model)

| Group | Example columns | What they contain |
|-------|-----------------|-------------------|
| **Identity** | `id`, `name`, `first`, `last`, `dob` | Who the person is. Never used as features — they identify individuals but carry no legitimate predictive meaning. |
| **COMPAS scores** | `decile_score` (1–10), `score_text` (Low/Medium/High), `v_decile_score` | The **algorithm's own risk predictions**. We deliberately don't use them — we're building our *own* predictor, not copying COMPAS. |
| **Current case details** | `c_jail_in`, `c_jail_out`, `c_offense_date`, `c_charge_desc`, `days_b_screening_arrest` | Dates and descriptions of the arrest that triggered the COMPAS screening. |
| **Recidivism details (`r_*`)** | `r_charge_degree`, `r_offense_date`, `r_charge_desc` | Details of the *new* crime — only filled in **if** the person re-offended. Mostly missing by design. |
| **Violent recidivism (`v_*`, `vr_*`)** | `is_violent_recid`, `vr_charge_desc` | Same, but specifically for *violent* re-offending (a stricter outcome we don't model). |
| **Study window** | `start`, `end`, `event` | When the observation period started/ended (in days). We use these to engineer `duration`. |
| **Age bucket** | `age_cat` | `Less than 25` / `25 - 45` / `Greater than 45` — a categorical version of `age` (redundant with `age`, so we use the number). |

💡 **Key idea:** a column being *in the file* doesn't mean it should be *in the model*. We exclude identifiers (meaningless), COMPAS's own scores (that's the tool we're auditing!), and leakage-prone columns like `is_recid` (it directly reveals the answer).


**What we're about to do:** Print the full list of column names so you can match them against the table above.


In [ ]:
# List every column in the raw file
print(f"The dataset has {data.shape[1]} columns:\n")
print(data.columns.tolist())


**What the output means:** All 53 raw column names. You can find each one in one of the groups from the table above. Next, let's look at the values inside the most important categorical columns.


In [ ]:
# Look at the possible values of the key categorical columns
print("Values of `race` and how many people are in each group:")
print(data["race"].value_counts(), "\n")

print("Values of `sex`:")
print(data["sex"].value_counts(), "\n")

print("Values of `c_charge_degree` (F = felony, M = misdemeanor):")
print(data["c_charge_degree"].value_counts(), "\n")

print("Values of the target `two_year_recid`:")
print(data["two_year_recid"].value_counts())


**What the output means:**
- **`race`**: African-American defendants are the largest group (~51%), followed by Caucasian (~34%). Some groups (Asian, Native American) have very few people — important later, because fairness statistics on tiny groups are unreliable.
- **`sex`**: roughly 81% male, 19% female.
- **`c_charge_degree`**: about 64% felonies, 36% misdemeanors.
- **`two_year_recid`**: the two classes (re-offended vs. not) are fairly balanced — we'll quantify this in the EDA section.


---
## 5. Data Shape and Structure

**What we're about to do:** Answer four basic "health check" questions about the dataset:
1. How many rows and columns are there?
2. What data type is each column (number, text, date)?
3. Where are values missing?
4. Are there duplicate rows?

These checks matter because almost every modeling problem (crashes, silent errors, biased results) starts with data-quality issues you didn't notice.


In [ ]:
# 1) How big is the dataset? .shape returns (rows, columns)
print(f"Rows (defendants): {data.shape[0]}")
print(f"Columns:           {data.shape[1]}")


**What the output means:** The dataset has **7,214 defendants** described by **53 columns**. That's a moderate size — big enough for meaningful statistics, small enough to train models in seconds.


In [ ]:
# 2) What type is each column?
# 'int64'/'float64' = numbers, 'object' = text (strings/categories)
data.dtypes.value_counts()


**What the output means:** Roughly half the columns are numeric (`int64`/`float64`) and half are text (`object`). Note that **dates are stored as text** here — pandas didn't automatically recognize them. That's fine for us, since we don't use the date columns directly.


In [ ]:
# 3) Missing values — count NaNs per column, show only columns that have any,
#    sorted from most missing to least
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"{len(missing)} of {data.shape[1]} columns contain missing values:\n")
missing


**What the output means:** Many columns have missing values, but look at *which* ones: they are almost all the `r_*`, `vr_*`, and jail-date columns. This missingness is **structural, not accidental**:
- `r_*` columns describe the *new* offense — they can only exist for people who re-offended.
- `vr_*` columns only exist for *violent* re-offenders (even rarer).
- `c_jail_in`/`c_jail_out` are missing when the person wasn't jailed.

✅ **The crucial check:** none of the **9 features our project uses** (`age`, the three `juv_*` counts, `priors_count`, `c_charge_degree`, `race`, `sex`, plus `start`/`end` for `duration`) appear in this list — our modeling columns are complete, so we need **no imputation** (no filling-in of missing values).


In [ ]:
# 4) Duplicate rows — are any defendants recorded twice?
print("Exact duplicate rows:", data.duplicated().sum())
print("Duplicate defendant ids:", data["id"].duplicated().sum())


**What the output means:** `0` duplicates on both checks — each row is a unique defendant. Duplicates would be dangerous because the same person could end up in both the training and test sets, making the model look better than it really is.


---
## 6. Creating Our Feature Set (as `Compas.py` does)

**What we're about to do:** Reproduce the exact feature-selection step from our project:

1. **Engineer** `duration = end − start` (days the person was observed in the study).
2. **Select** the 9 feature columns + the target.

From here on we work with this smaller, clean table — just like the model does.


In [ ]:
# Step 1: engineer the observation-window length, exactly as Compas.py does
data["duration"] = data["end"] - data["start"]

# Step 2: keep only the columns our project uses
features = ["age", "juv_fel_count", "juv_misd_count", "juv_other_count",
            "priors_count", "c_charge_degree", "race", "sex", "duration"]
target = "two_year_recid"

df = data[features + [target]].copy()   # our working table

print("Working table shape:", df.shape)
df.head()


**What the output means:** We now have a tidy table: **7,214 rows × 10 columns** (9 features + 1 target). This is *exactly* what the model sees (before scaling/encoding). Every analysis below uses this table.

> 📝 **Side note for later reading:** ProPublica's own analysis additionally *filtered rows* (e.g. keeping only cases where the COMPAS screening happened within 30 days of arrest, `days_b_screening_arrest` between −30 and 30). Our project uses **all 7,214 rows** without that filter. Knowing this difference matters when comparing our numbers to published papers.


---
## 7. Exploratory Data Analysis (EDA)

EDA means **looking at the data before modeling** — computing summaries and drawing pictures to understand distributions, spot oddities, and build intuition. We'll cover:

7.1 Summary statistics
7.2 Target variable distribution (class balance)
7.3 Distributions of the important numeric features
7.4 Categorical features vs. the target
7.5 Correlation analysis
7.6 Interesting patterns & observations


### 7.1 Summary Statistics

**What we're about to do:** Use `.describe()` to get, for every numeric column: the count, mean, standard deviation (spread), minimum, maximum, and the quartiles (25%, 50% = median, 75%).


In [ ]:
# Summary statistics for the numeric features (rounded for readability)
df.describe().round(2)


**What the output means (reading the table):**
- **`age`**: ranges 18–96, average ≈ 35, median 31 → the population skews young.
- **`priors_count`**: median is 2 but max is 38 → most people have few priors, a small tail has very many. This kind of "long right tail" is called **right-skew**.
- **`juv_*` counts**: the 75th percentile is 0 for all three → the vast majority of defendants have **zero** juvenile records; these features are mostly zeros with rare positive values.
- **`duration`**: mean ≈ 542 days, max 1,186. Curiously, the minimum is **−256** — a handful of rows have `end` before `start`. Tiny data quirks like this are normal in real datasets and worth noticing.
- **`two_year_recid`**: the mean of a 0/1 column is the **fraction of 1s** → ≈ 0.45, meaning about **45% of defendants re-offended** within two years.


### 7.2 Target Variable Distribution (Class Imbalance Analysis)

**What we're about to do:** Count and plot how many people re-offended (1) vs. not (0). If one class hugely outnumbered the other (say 95% vs 5%), a lazy model could score 95% accuracy by always predicting the majority — so we always check balance first.


In [ ]:
# Count each class and convert to percentages
counts = df[target].value_counts().sort_index()
pcts = (counts / len(df) * 100).round(1)

print("Class counts:")
print(f"  0 = did NOT re-offend: {counts[0]:>5} people ({pcts[0]}%)")
print(f"  1 = DID re-offend:     {counts[1]:>5} people ({pcts[1]}%)")

# Bar chart of the target distribution
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Did not re-offend (0)", "Re-offended (1)"],
              counts.values, color=[C_NO, C_YES], width=0.6)
# Direct labels on top of each bar (so no one has to guess the heights)
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f"{bar.get_height():,.0f}  ({pct}%)", ha="center", fontsize=11)
ax.set_ylabel("Number of defendants")
ax.set_title("Target: two-year recidivism")
ax.set_ylim(0, counts.max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()


**What the output means:** About **55% did not re-offend vs. 45% did**. This is a **mildly imbalanced but essentially healthy** split:
- ✅ Plain accuracy is still a meaningful metric (the "always predict 0" baseline gets only ~55%).
- ✅ No need for special imbalance techniques (oversampling, class weights).

Compare this with the German Credit dataset (70/30), where imbalance is a bigger deal.


### 7.3 Distributions of Important Numeric Features

**What we're about to do:** Draw **histograms** for `age`, `priors_count`, and `duration`. A histogram chops the number line into bins and counts how many people fall in each bin — it shows the *shape* of a feature.


In [ ]:
# Histograms of the three key numeric features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["age"], bins=30, color=C_MAIN, edgecolor="white")
axes[0].set_title("Age")
axes[0].set_xlabel("Years")
axes[0].set_ylabel("Number of defendants")

axes[1].hist(df["priors_count"], bins=39, color=C_MAIN, edgecolor="white")
axes[1].set_title("Prior offenses (priors_count)")
axes[1].set_xlabel("Count of prior crimes")

axes[2].hist(df["duration"], bins=30, color=C_MAIN, edgecolor="white")
axes[2].set_title("Observation duration")
axes[2].set_xlabel("Days observed (end − start)")

sns.despine()
plt.tight_layout()
plt.show()


**What the output means:**
- **Age** is right-skewed: a big bulge in the 20s–30s, tapering off toward older ages. Criminal-justice populations are typically young.
- **Priors** is *extremely* right-skewed: a huge spike at 0–2 priors, with a long thin tail out to 38. Features like this often dominate a linear model.
- **Duration** is spread widely, from slightly negative (the small end-before-start quirks we spotted) up to ~1,186 days, with a large group at short durations. Keep it in mind for the correlation section — short observation windows turn out to be strongly tied to re-offending.

**Why we care about skew:** our pipeline applies `StandardScaler` (subtract mean, divide by standard deviation). Scaling doesn't remove skew, but it puts all features on comparable ranges so no feature dominates just because its numbers are bigger.


**What we're about to do next:** Check how age and priors relate to the *target* by splitting each histogram into the two outcome groups. This previews which features carry predictive signal.


In [ ]:
# Compare feature distributions for re-offenders vs non-re-offenders
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for value, color, label in [(0, C_NO, "Did not re-offend"),
                            (1, C_YES, "Re-offended")]:
    subset = df[df[target] == value]
    axes[0].hist(subset["age"], bins=30, alpha=0.6, color=color,
                 label=label, edgecolor="white")
    axes[1].hist(subset["priors_count"], bins=39, alpha=0.6, color=color,
                 label=label, edgecolor="white")

axes[0].set_title("Age, split by outcome")
axes[0].set_xlabel("Years"); axes[0].set_ylabel("Number of defendants")
axes[0].legend()
axes[1].set_title("Prior offenses, split by outcome")
axes[1].set_xlabel("Count of prior crimes")
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

# Numeric confirmation: average feature values per outcome group
df.groupby(target)[["age", "priors_count", "juv_fel_count", "duration"]].mean().round(2)


**What the output means:**
- **Age:** the orange (re-offended) group is concentrated at *younger* ages; the blue group has more mass at older ages. Average age: ~37 for non-re-offenders vs. ~33 for re-offenders.
- **Priors:** re-offenders have clearly more priors on average (~5.0 vs. ~2.3). This is one of the model's strongest legitimate signals.
- The `groupby(target).mean()` table confirms it with numbers: every history-related feature is higher in the re-offended group.


### 7.4 Categorical Features vs. the Target

**What we're about to do:** For each categorical feature (`race`, `sex`, `c_charge_degree`), compute the **recidivism rate** (fraction of 1s) within each category. This is where fairness questions start appearing, so read these plots carefully.


In [ ]:
# Recidivism rate (mean of the 0/1 target) per category
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col in zip(axes, ["race", "sex", "c_charge_degree"]):
    rates = df.groupby(col)[target].mean().sort_values(ascending=False)
    bars = ax.barh(rates.index[::-1], rates.values[::-1] * 100, color=C_MAIN)
    for bar in bars:
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.0f}%", va="center", fontsize=10)
    ax.axvline(df[target].mean() * 100, color="gray", linestyle="--",
               linewidth=1, label="overall rate")
    ax.set_title(f"Recidivism rate by {col}")
    ax.set_xlabel("% who re-offended within 2 years")
    ax.set_xlim(0, 80)
    ax.legend(loc="lower right", fontsize=9)

sns.despine()
plt.tight_layout()
plt.show()


**What the output means:** The dashed gray line is the overall rate (~45%). Categories to the right of it re-offend more often than average:
- **Race:** among the large groups, African-American defendants have the highest observed rate (~51%) vs. Caucasian ~39%. The Native American bar is actually highest (~56%) and the Asian bar lowest (~28%), but those groups have only 18 and 32 people — tiny groups give very noisy percentages, so don't over-read them.
- **Sex:** men (~47%) re-offend more often than women (~36%).
- **Charge degree:** felony defendants (~49%) more than misdemeanor (~38%).

⚠️ **Critical interpretation warning:** these are **observed differences in recorded arrests**, not truths about groups of people. Arrest data reflects policing patterns — if one neighborhood is policed more heavily, its residents get *recorded* as re-offending more often for the same behavior. This is exactly the kind of **historical bias** a model can learn and amplify — the core motivation of our project. More in Section 8.


### 7.5 Correlation Analysis

**What we're about to do:** Compute the **correlation matrix** of the numeric columns. Correlation ranges from −1 to +1:
- **+1** = the two variables rise together perfectly
- **0** = no linear relationship
- **−1** = one rises exactly as the other falls

We visualize it as a heatmap. We use a **diverging** colormap (two hues meeting at a neutral middle) because correlation has a natural zero point — red = positive, blue = negative.


In [ ]:
# Correlation matrix of numeric features + target
numeric_cols = ["age", "juv_fel_count", "juv_misd_count", "juv_other_count",
                "priors_count", "duration", target]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, square=True,
            cbar_kws={"label": "correlation"}, ax=ax)
ax.set_title("Correlation between numeric features and the target")
plt.tight_layout()
plt.show()


**What the output means (focus on the last row/column, correlations with `two_year_recid`):**
- **`duration` ≈ −0.78** — by far the strongest correlation, and it deserves suspicion! People observed longer re-offended less — but this is largely **mechanical**: re-offending can end your observation window early, so short `duration` is partly a *consequence* of the outcome, not a cause. Features that encode the answer like this are called **label leakage**. It's an honest and important observation about our project's feature set: `duration` likely inflates accuracy.
- **`priors_count` ≈ +0.28** — the strongest *legitimate* predictor: more priors → more likely to re-offend.
- **`age` ≈ −0.19** — negative: older defendants re-offend *less*.
- **`juv_*` counts** — small positive correlations (~0.09–0.11); weak individually because most values are zero.

Also note `priors_count` correlates with the juvenile counts (people with juvenile records accumulate adult priors) — mild **redundancy** between features, which logistic regression tolerates fine.

**Why no correlation for categoricals?** Pearson correlation needs numbers. For `race`/`sex`/`c_charge_degree` we used group rates (7.4) instead — the right tool for categories.


### 7.6 Interesting Patterns & Observations

**What we're about to do:** One last cross-view: how do **age and priors interact with race**? This matters because if protected groups differ on legitimate features, a model can partially *infer* race even without being given it — so-called **proxy features**.


In [ ]:
# Do the two strongest predictors differ across racial groups?
# (Focus on the two largest groups; small groups are statistically noisy.)
summary = (df.groupby("race")
             .agg(n=("race", "size"),
                  avg_age=("age", "mean"),
                  avg_priors=("priors_count", "mean"),
                  recid_rate=(target, "mean"))
             .round(2)
             .sort_values("n", ascending=False))
summary


**What the output means:** African-American defendants in this dataset are on average **younger** and have **more priors** than Caucasian defendants. Since young age and high priors both predict recidivism, these features act as **partial proxies for race**: even a model that never sees the `race` column can produce racially disparate predictions. This is why "just delete the protected column" (*fairness through unawareness*) does **not** guarantee fairness — and why our project instead measures fairness explicitly and penalizes it in the loss function.

**EDA takeaways so far:**
1. Clean data: no missing values or duplicates in our modeling columns (aside from a few negative `duration` quirks).
2. Mild class imbalance (55/45) — accuracy is usable.
3. `priors_count` (+0.28) and `age` (−0.19) are the main legitimate signals; `duration` (−0.78) is suspiciously strong and partly leaks the label.
4. Recidivism rates differ visibly across race and sex — the fairness problem is *in the data*, before any model exists.
5. Proxy features mean fairness must be measured, not assumed.


---
## 8. Fairness-Related Analysis

### 8.1 What is a protected attribute?

A **protected attribute** is a characteristic that laws and ethics say should *not* drive decisions about a person — e.g. race, sex, age, religion, national origin. In this dataset:

| Attribute | Present as | Used in our project as |
|-----------|-----------|------------------------|
| **Race** | `race` column | ⭐ the **sensitive feature** — all group-fairness metrics compare racial groups |
| Sex | `sex` column | a model feature (a common and debatable choice worth being aware of!) |
| Age | `age` column | a model feature (age is protected in employment law, but is a standard criminological predictor) |

### 8.2 Why race, and why does it matter here?

ProPublica's 2016 analysis of exactly this data found that COMPAS made **different kinds of mistakes for different races**:
- Black defendants who did *not* re-offend were nearly **twice as likely to be falsely labeled high-risk** (false positives).
- White defendants who *did* re-offend were more likely to be **falsely labeled low-risk** (false negatives).

A false positive here can mean harsher bail terms or a longer sentence for someone who would not have re-offended. That's why this dataset became *the* benchmark for fairness research.

### 8.3 How our project uses the protected attribute

In `Compas.py`, the `race` column is kept aside as `sensitive_features` and passed to the fairness metrics:

- **Group fairness** (from `GroupFairness.py`) — compares model behavior *between groups*:
  - **Demographic parity difference** — do all races receive positive (high-risk) predictions at the same rate?
  - **Equalized odds difference** — are error rates (false positives & false negatives) equal across races? *(This is precisely the failure ProPublica exposed.)*
  - **Equal opportunity difference** — among people who truly re-offend, are all races equally likely to be correctly identified?
  - **Disparate impact** — the *ratio* of positive-prediction rates between groups (a legal standard: below 0.8 is often considered discriminatory).
- **Individual fairness** (from `IndividualFairness.py`) — Theil index, generalized entropy, Atkinson, Gini — measures whether prediction *benefits/errors* are spread evenly across individuals, regardless of group.

The custom loss blends these with accuracy: `total = α·BCE + (1−α)·(β·group + (1−β)·individual)`.

### 8.4 Sources of bias in this dataset (know them by name)

| Bias type | How it shows up in COMPAS data |
|-----------|-------------------------------|
| **Historical / measurement bias** | The target is *re-arrest*, not *re-offense*. Heavier policing of Black neighborhoods → more recorded arrests for the same underlying behavior → the "ground truth" itself is biased. |
| **Representation bias** | One county (Broward, FL), 2013–2014. Small Asian/Native American groups make their fairness statistics unstable. |
| **Proxy bias** | `priors_count`, `age`, and even `duration` correlate with race (Section 7.6), so race leaks into predictions through legitimate-looking features. |
| **Label imbalance across groups** | Base rates differ by group (7.4), which makes several fairness definitions *mathematically impossible to satisfy simultaneously* (the famous impossibility result from the COMPAS debate). |

**What we're about to do:** Compute the **base rate** (actual recidivism rate) per racial group and the group sizes — the two numbers every fairness metric is built on.


In [ ]:
# Base rates: the raw material of every group-fairness metric
base = (df.groupby("race")[target]
          .agg(n="size", base_rate="mean")
          .sort_values("n", ascending=False))
base["base_rate"] = (base["base_rate"] * 100).round(1)

print(base, "\n")

# The gap between the two largest groups
gap = (df[df.race == "African-American"][target].mean()
       - df[df.race == "Caucasian"][target].mean()) * 100
print(f"Base-rate gap, African-American vs Caucasian: {gap:.1f} percentage points")


**What the output means:** The two largest groups differ in observed base rate by roughly **12 percentage points**. This single number explains much of the fairness dilemma:

- If a model predicts *accurately*, its positive-prediction rates will differ across groups → it violates **demographic parity**.
- If we force equal prediction rates, the model must make *more errors* in some group → it can violate **equalized odds** or lose accuracy.

There is no setting that satisfies everything at once — which is exactly why our project **sweeps α and β** to map the whole trade-off curve instead of picking one "correct" point.


---
## 9. Step-by-Step Workflow — From Raw Data to Model-Ready Tensors

This section reproduces, step by step, the exact preprocessing pipeline in `Compas.py`, explaining **why each step exists**. Order matters!

| Step | What | Why |
|------|------|-----|
| 1 | Load data | Get the raw table |
| 2 | Engineer `duration` | Add observation-window information |
| 3 | Select features + target | Drop identifiers, leaky columns, COMPAS's own scores |
| 4 | Split train / validation / test (60/20/20) | Honest evaluation on data the model never saw |
| 5 | Set aside the sensitive attribute (`race`) | Needed later to *measure* fairness on each split |
| 6 | Scale numeric + one-hot encode categorical features | Models need comparable numeric inputs |
| 7 | Convert to PyTorch tensors | The training code speaks PyTorch |


**What we're about to do (Step 4):** Split the data. We split **twice**: first 80/20 into train+val vs. **test**, then the 80% again 75/25 into **train** vs. **validation**. Net result: 60% train / 20% validation / 20% test. `random_state=42` makes the split reproducible — everyone who runs this gets the identical split.


In [ ]:
from sklearn.model_selection import train_test_split

X = df[features]   # the 9 feature columns
y = df[target]     # the 0/1 labels

# First split: 80% (train+val) vs 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Second split: of the 80%, take 25% for validation -> 60/20/20 overall
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42)

print(f"Train:      {X_train.shape[0]:>5} rows ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"Validation: {X_val.shape[0]:>5} rows ({X_val.shape[0]/len(df)*100:.0f}%)")
print(f"Test:       {X_test.shape[0]:>5} rows ({X_test.shape[0]/len(df)*100:.0f}%)")


**What the output means:** ~4,328 training rows, ~1,443 each for validation and test. The **test set is locked away** — it's only used at the very end to report honest performance. The validation set is for tuning choices during development.


**What we're about to do (Step 5):** Save the `race` column of each split separately. It's used as a *feature* too, but the fairness metrics need it in its **original readable form** (one-hot encoding would smear it across several 0/1 columns).


In [ ]:
# Keep the sensitive attribute in readable form for fairness metrics
sensitive_train = X_train["race"]
sensitive_val   = X_val["race"]
sensitive_test  = X_test["race"]

print("Racial groups in the TEST split (fairness metrics are computed on this):")
print(sensitive_test.value_counts())


**What the output means:** The test split preserves roughly the full data's racial mix. ⚠️ Notice how small the smallest groups get (a handful of people) — one individual flipping outcome can swing that group's rate by several points. That's why serious fairness reporting focuses on the larger groups.


**What we're about to do (Step 6):** Build the preprocessing transformer:
- **`StandardScaler`** for numeric columns: transforms each to mean 0, standard deviation 1. Without it, `duration` (0–1,186) would numerically dwarf `juv_fel_count` (0–20) and gradient descent would struggle.
- **`OneHotEncoder`** for categorical columns: turns e.g. `race` into several 0/1 columns (`race_Caucasian`, `race_Hispanic`, …). `drop='first'` drops one category per feature — the "reference category" — because it's redundant (if all others are 0, you're in the reference group).

🚨 **Golden rule:** `fit_transform` on **train only**, then plain `transform` on validation/test. The scaler must learn its means/standard-deviations **only from training data** — otherwise information from the test set "leaks" into training and inflates results.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Detect column types automatically (same as Compas.py)
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numeric:    ", numeric_features)
print("Categorical:", categorical_features)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),          # scale numbers
    ("cat", OneHotEncoder(drop="first"), categorical_features),  # encode categories
])

# FIT on train only; APPLY to all three splits
X_train_p = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

print(f"\nColumns before preprocessing: {X_train.shape[1]}")
print(f"Columns after preprocessing:  {X_train_p.shape[1]}")


**What the output means:** 9 human-readable columns became **13 numeric columns**: 6 scaled numbers + 7 one-hot columns — `c_charge_degree` contributes 1 (2 categories − 1 dropped), `race` contributes 5 (6 − 1), and `sex` contributes 1 (2 − 1). This 13-column matrix is the model's actual input — `input_dim = 13` in the project.


**What we're about to do (Step 7):** Convert the matrices to **PyTorch tensors** — the array format PyTorch trains on. Labels get reshaped to a column (`view(-1, 1)`) because the model outputs one probability per row.


In [ ]:
import torch

X_train_t = torch.tensor(X_train_p, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_p,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test_p,  dtype=torch.float32)

y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val_t   = torch.tensor(y_val.values,   dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.float32).view(-1, 1)

print("Feature tensor:", X_train_t.shape, "| Label tensor:", y_train_t.shape)
print("\n✅ Data is now exactly in the form Compas.py feeds to the logistic-regression model.")


**What the output means:** Training tensors of shape `[4328, 13]` (features) and `[4328, 1]` (labels). From here, `Compas.py` trains a logistic-regression model (a single linear layer + sigmoid) for each of the **196 combinations** of α, β, group-fairness metric, and individual-fairness metric, then writes `results.csv` and heatmaps.


---
## 10. Summary — What We Learned

### The dataset in one paragraph
The COMPAS dataset contains **7,214 criminal defendants** from Broward County, Florida (2013–2014), each described by demographics, criminal history, and current charge, with a binary target **`two_year_recid`** recording whether they were re-arrested within two years. It is the landmark dataset of algorithmic-fairness research because ProPublica showed the COMPAS tool's errors fell disproportionately on Black defendants.

### Key insights from our analysis

1. **Data quality is excellent** for our purposes: no missing values or duplicates in the 9 modeling features (heavy missingness elsewhere is structural — `r_*` columns only exist for re-offenders).
2. **Classes are nearly balanced** (55% no / 45% yes), so plain accuracy is a fair headline metric.
3. **Strongest legitimate signals:** more priors → more recidivism (r ≈ +0.28); older age → less (r ≈ −0.19). The engineered `duration` feature correlates at −0.78 but partly *leaks the label* (re-offending shortens the observation window) — an important caveat about the feature set.
4. **Fairness problem is visible before modeling:** observed recidivism rates differ by ~12 percentage points between African-American and Caucasian defendants — partly reflecting policing/measurement bias, not just behavior.
5. **Proxy features exist:** age and priors differ across races, so removing the `race` column would *not* remove racial disparity from predictions.
6. **Preprocessing pipeline:** engineer `duration` → select 9 features → 60/20/20 split → keep `race` aside for fairness metrics → scale + one-hot (fit on train only!) → 13-dim tensors.

### How this prepares us for the next stage

With the data understood, the project trains a **logistic-regression model with a composite loss**:

`total = α·BCE + (1−α)·(β·group_fairness + (1−β)·individual_fairness)`

sweeping **α, β ∈ {0, 0.005, 0.05, 0.25, 0.5, 0.75, 1}** across 4 group-fairness × 4 individual-fairness metrics (196 combos), then visualizing accuracy and fairness as α×β heatmaps. Everything you saw here — the base-rate gap, the proxy features, the class balance — is what those heatmaps are ultimately measuring the model against.

📗 **Next:** open the companion **German Credit Dataset notebook**, which repeats this exact study for the second dataset (protected attribute: **sex**), so the two experiments can be compared side by side.
